# Population by class and boundary

Combine a population raster with a categorical raster to estimate population totals and proportions by class within each boundary.


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.warp import reproject

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
DATA = ROOT / 'data_raw'
OUTPUTS = ROOT / 'outputs'
TABLES = ROOT / 'tables'
for folder in (OUTPUTS, TABLES):
    folder.mkdir(parents=True, exist_ok=True)

BOUNDARY_FILE = DATA / 'boundary.gpkg'
POPULATION_RASTER = DATA / 'population.tif'
CLASS_RASTER = OUTPUTS / 'categorical_raster_reclassified.tif'
BOUNDARY_ID_FIELD = None
CLASS_LABELS = {}
ALL_TOUCHED = False

for path in (BOUNDARY_FILE, POPULATION_RASTER, CLASS_RASTER):
    if not path.exists():
        raise FileNotFoundError(f'Required input not found: {path}')

boundaries = gpd.read_file(BOUNDARY_FILE)
with rasterio.open(POPULATION_RASTER) as pop_src, rasterio.open(CLASS_RASTER) as class_src:
    if boundaries.crs != pop_src.crs:
        boundaries = boundaries.to_crs(pop_src.crs)
    population = pop_src.read(1, masked=False).astype('float64')
    if pop_src.nodata is not None:
        population = np.where(population == pop_src.nodata, 0.0, population)
    population = np.where(np.isfinite(population), population, 0.0)

    nodata_class = class_src.nodata if class_src.nodata is not None else -9999
    aligned_class = np.full(population.shape, nodata_class, dtype='int16')
    reproject(source=rasterio.band(class_src, 1), destination=aligned_class, src_transform=class_src.transform, src_crs=class_src.crs, src_nodata=class_src.nodata, dst_transform=pop_src.transform, dst_crs=pop_src.crs, dst_nodata=nodata_class, resampling=Resampling.nearest)

    valid_codes = sorted(int(x) for x in pd.unique(aligned_class.ravel()) if x != nodata_class)
    rows = []
    for i, geom in enumerate(boundaries.geometry):
        boundary_id = str(boundaries.loc[i, BOUNDARY_ID_FIELD]) if BOUNDARY_ID_FIELD and BOUNDARY_ID_FIELD in boundaries.columns else f'boundary_{i+1:03d}'
        boundary_mask = rasterize([(geom, 1)], out_shape=population.shape, transform=pop_src.transform, fill=0, all_touched=ALL_TOUCHED, dtype='uint8').astype(bool)
        boundary_total = float(population[boundary_mask].sum())
        for code in valid_codes:
            class_total = float(population[boundary_mask & (aligned_class == code)].sum())
            rows.append({'boundary_id': boundary_id, 'class_code': int(code), 'class_label': CLASS_LABELS.get(int(code), 'unlabelled'), 'population_total': class_total, 'boundary_population_total': boundary_total, 'population_proportion': class_total / boundary_total if boundary_total > 0 else np.nan})

result = pd.DataFrame(rows)
closure = result.groupby('boundary_id', as_index=False).agg(class_population_sum=('population_total', 'sum'), boundary_population_total=('boundary_population_total', 'first'))
closure['absolute_difference'] = (closure['class_population_sum'] - closure['boundary_population_total']).abs()
result.to_csv(TABLES / 'population_by_class_and_boundary.csv', index=False)
closure.to_csv(TABLES / 'population_class_closure_check.csv', index=False)
with open(OUTPUTS / 'population_by_class_summary.json', 'w', encoding='utf-8') as f:
    json.dump({'run_time_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'), 'all_touched': ALL_TOUCHED, 'n_boundaries': int(result['boundary_id'].nunique()) if not result.empty else 0, 'n_classes': int(result['class_code'].nunique()) if not result.empty else 0, 'max_closure_difference': float(closure['absolute_difference'].max()) if not closure.empty else 0.0}, f, indent=2)
result
